# Object_Detection Phase 5 — YOLOv11 파인튜닝

**목적**: AI Hub 음식이미지 데이터 전체(800종)로 yolo11s.pt 파인튜닝 → 한식/지방특산물 직접 탐지

**대상 데이터셋**: AI Hub '비전영역, 음식이미지 및 정보소개 텍스트 데이터' (dataSetSn=71564)  
- 총 232,087장 / 800종 / 4대분류 (특수외식 35% / 일반외식배달 34.7% / 끼니대체 26.4% / 음료차류 3.9%)
- 라벨: JSON 바운딩박스(2d_annotation) 포함

**결과물**: `best.pt` → 로컬 `Object_Detection/models/korean_food_v1_best.pt`

---

### 파일 키 목록 (AI Hub 다운로드 페이지에서 확인)

| 파일 | 키 | 크기 | 분류 |
|------|----|------|------|
| TS.z01 | 502331 | 100 GB | 학습 이미지 (분할 1) |
| TS.z02 | 502332 | 100 GB | 학습 이미지 (분할 2) |
| TS.z03 | 502333 | 100 GB | 학습 이미지 (분할 3) |
| TS.z04 | 502334 | 100 GB | 학습 이미지 (분할 4) |
| TS.z05 | 502335 | 100 GB | 학습 이미지 (분할 5) |
| TS.z06 | 502336 | 100 GB | 학습 이미지 (분할 6) |
| TS.z07 | 502337 | 100 GB | 학습 이미지 (분할 7) |
| TS.zip | 502338 | 39.51 GB | 학습 이미지 (분할 마지막) |
| VS.zip | 502340 | 88.80 GB | 검증 이미지 |
| VL.zip | 502341 | **17.77 MB** | 검증 라벨 JSON ← 먼저 다운로드 |

> **전략**: 총 용량 ~839GB — Colab 디스크(~100GB)에 전부 올릴 수 없음.  
> 학습 이미지 분할 파일을 **1개씩 받아 처리 후 삭제**하는 방식으로 진행.

### 실행 순서
1. 런타임 유형 → GPU(A100) 설정
2. Step 0: 환경 설정 + Drive 마운트
3. Step 0.5: AI Hub API 키 입력 → 자동 다운로드
4. Step 1~7: 변환 → 학습 → 검증 → best.pt 다운로드

## Step 0 — 환경 설정 및 Drive 마운트

In [ ]:
# GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
print('GPU 이름:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT    = '/content/drive/MyDrive'
DATASET_DIR   = f'{DRIVE_ROOT}/aihub_food'        # 다운로드된 원본 데이터 저장 위치
FINETUNE_DIR  = f'{DRIVE_ROOT}/finetune_dataset'   # 변환된 YOLO 데이터셋 저장
RUNS_DIR      = f'{DRIVE_ROOT}/runs'               # 학습 결과 저장
DOWNLOAD_TMP  = '/content/aihub_tmp'               # Colab 로컬 임시 다운로드 공간

for d in [DATASET_DIR, FINETUNE_DIR, RUNS_DIR, DOWNLOAD_TMP,
          f'{FINETUNE_DIR}/train/images', f'{FINETUNE_DIR}/train/labels',
          f'{FINETUNE_DIR}/val/images',   f'{FINETUNE_DIR}/val/labels']:
    os.makedirs(d, exist_ok=True)

print('Drive 마운트 완료')
print(f'원본 데이터:   {DATASET_DIR}')
print(f'YOLO 데이터셋: {FINETUNE_DIR}')
print(f'학습 결과:     {RUNS_DIR}')
print(f'임시 다운로드: {DOWNLOAD_TMP}')

# 디스크 여유 공간 확인
!df -h /content | tail -1
!df -h /content/drive/MyDrive | tail -1

In [ ]:
!pip install ultralytics -q
print('ultralytics 설치 완료')

## Step 0.5 — AI Hub Open API 다운로드

> AI Hub 로그인 후 **마이페이지 → API 키 발급** 에서 키를 복사해 붙여넣는다.  
> [AI Hub Open API 이용안내](https://aihub.or.kr/intrcn/guid/usagePatter.do)

### 다운로드 전략
- **VL.zip (검증 라벨, 17.77MB)**: 즉시 전체 다운로드 → Drive 영구 보관
- **TS.z01~TS.zip (학습 이미지, 각 100GB)**: 분할 파일 1개씩 Colab 로컬에 받고 → 이미지 추출 → Drive로 이동 → 삭제 반복
- **VS.zip (검증 이미지, 88.80GB)**: 선택 다운로드 (Drive 여유 공간 확인 후)

In [ ]:
import requests
import os
import time
from pathlib import Path

# ─────────────────────────────────────────────────────
# AI Hub API 키 입력
# 마이페이지 → API 키 발급 → 복사 후 아래 붙여넣기
# ─────────────────────────────────────────────────────
AIHUB_API_KEY = input('AI Hub API 키를 입력하세요: ').strip()

# 파일 키 목록 (AI Hub 다운로드 페이지 기준)
FILE_KEYS = {
    'TS.z01': 502331,   # 학습 이미지 분할 1 (100GB)
    'TS.z02': 502332,   # 학습 이미지 분할 2 (100GB)
    'TS.z03': 502333,   # 학습 이미지 분할 3 (100GB)
    'TS.z04': 502334,   # 학습 이미지 분할 4 (100GB)
    'TS.z05': 502335,   # 학습 이미지 분할 5 (100GB)
    'TS.z06': 502336,   # 학습 이미지 분할 6 (100GB)
    'TS.z07': 502337,   # 학습 이미지 분할 7 (100GB)
    'TS.zip': 502338,   # 학습 이미지 분할 마지막 (39.51GB)
    'VS.zip': 502340,   # 검증 이미지 (88.80GB)
    'VL.zip': 502341,   # 검증 라벨 JSON (17.77MB) ← 가장 먼저
}

API_BASE = 'https://api.aihub.or.kr/down/0.do'


def download_aihub_file(
    file_name: str,
    file_key: int,
    save_dir: str,
    api_key: str,
    chunk_size: int = 8 * 1024 * 1024,   # 8MB 청크
) -> str:
    """
    AI Hub Open API를 통해 파일 1개를 다운로드한다.
    이미 존재하면 스킵.
    Returns: 저장된 파일 경로
    """
    save_path = Path(save_dir) / file_name
    if save_path.exists():
        size_gb = save_path.stat().st_size / 1024 ** 3
        print(f'✅ 이미 존재: {file_name} ({size_gb:.2f} GB) — 스킵')
        return str(save_path)

    url = f'{API_BASE}?seq={file_key}'
    headers = {'Authorization': api_key}

    print(f'다운로드 시작: {file_name} (key={file_key})')
    start = time.time()

    try:
        resp = requests.get(url, headers=headers, stream=True, timeout=60)
        resp.raise_for_status()

        total = int(resp.headers.get('Content-Length', 0))
        downloaded = 0

        with open(save_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total > 0:
                        pct = downloaded / total * 100
                        gb  = downloaded / 1024 ** 3
                        print(f'  {pct:5.1f}%  {gb:.2f} GB', end='\r')

        elapsed = time.time() - start
        size_gb = save_path.stat().st_size / 1024 ** 3
        print(f'\n✅ 완료: {file_name} | {size_gb:.2f} GB | {elapsed/60:.1f}분')
        return str(save_path)

    except requests.HTTPError as e:
        print(f'\n❌ HTTP 오류 {e.response.status_code}: {file_name}')
        if e.response.status_code == 401:
            print('   → API 키를 확인하세요 (마이페이지 → API 키 재발급)')
        elif e.response.status_code == 403:
            print('   → 데이터 신청 승인 여부를 확인하세요')
        raise


print(f'API 키 설정 완료 (길이: {len(AIHUB_API_KEY)}자)')
print(f'다운로드 대상 파일: {len(FILE_KEYS)}개')

In [ ]:
import zipfile
import subprocess

# ─────────────────────────────────────────────────────
# [즉시 실행] VL.zip — 검증 라벨 JSON (17.77 MB)
# 가장 먼저 Drive에 보관 → Step 1 클래스 수집에 활용
# ─────────────────────────────────────────────────────
vl_zip_path = download_aihub_file(
    file_name='VL.zip',
    file_key=FILE_KEYS['VL.zip'],
    save_dir=DATASET_DIR,     # Drive에 직접 저장 (영구 보관)
    api_key=AIHUB_API_KEY,
)

# 압축 해제
vl_extract_dir = f'{DATASET_DIR}/val_labels'
os.makedirs(vl_extract_dir, exist_ok=True)

if not list(Path(vl_extract_dir).rglob('*.json')):
    print(f'VL.zip 압축 해제 중 → {vl_extract_dir}')
    with zipfile.ZipFile(vl_zip_path, 'r') as zf:
        zf.extractall(vl_extract_dir)
    json_count = len(list(Path(vl_extract_dir).rglob('*.json')))
    print(f'✅ 압축 해제 완료 — JSON {json_count:,}개')
else:
    json_count = len(list(Path(vl_extract_dir).rglob('*.json')))
    print(f'✅ 이미 해제됨 — JSON {json_count:,}개')

In [ ]:
# ─────────────────────────────────────────────────────
# 학습 이미지 분할 다운로드 + 이미지 추출 (1개씩 처리)
#
# 전략:
#   1. Colab 로컬(/content/aihub_tmp)에 분할 파일 1개 다운로드
#   2. 이미지 파일(.jpg/.png)만 Drive(DATASET_DIR/train_images)로 이동
#   3. 로컬 임시 파일 삭제 → 다음 분할 파일 진행
#
# TS.z01~TS.z07 + TS.zip 은 7z 분할 압축 포맷
# → 전체 파트를 모두 받아야 완전히 해제 가능
# → 대신 각 파트에서 jpg만 스트리밍 추출 가능
# ─────────────────────────────────────────────────────

!apt-get install -y p7zip-full -q   # 7z 분할 압축 해제 도구

TRAIN_IMG_DIR = f'{DATASET_DIR}/train_images'
os.makedirs(TRAIN_IMG_DIR, exist_ok=True)

# 다운로드할 분할 파일 목록 (순서 중요: z01부터)
TRAIN_PARTS = [
    ('TS.z01', 502331),
    ('TS.z02', 502332),
    ('TS.z03', 502333),
    ('TS.z04', 502334),
    ('TS.z05', 502335),
    ('TS.z06', 502336),
    ('TS.z07', 502337),
    ('TS.zip', 502338),
]

# Colab 로컬 디스크 여유 확인
result = subprocess.run(['df', '-BG', '/content'], capture_output=True, text=True)
print('Colab 로컬 디스크 현황:')
print(result.stdout)

# ⚠️ 모든 분할 파일을 한 번에 다운로드하면 디스크 부족 가능
# 아래 셀에서 한 번에 1개씩 처리

In [ ]:
# ─────────────────────────────────────────────────────
# 분할 파일 1개 처리 함수
# ─────────────────────────────────────────────────────
def process_one_part(file_name: str, file_key: int):
    """
    분할 파일 1개를 Colab 로컬에 다운로드 → 이미지 추출 → Drive 이동 → 로컬 삭제
    """
    local_path = f'{DOWNLOAD_TMP}/{file_name}'

    # 이미 Drive에 이미지가 충분히 있으면 스킵
    existing = len(list(Path(TRAIN_IMG_DIR).rglob('*.jpg')))
    print(f'\n현재 Drive train_images: {existing:,}장')

    # 1. 다운로드 (로컬)
    download_aihub_file(
        file_name=file_name,
        file_key=file_key,
        save_dir=DOWNLOAD_TMP,
        api_key=AIHUB_API_KEY,
    )

    # 2. 이미지 추출 → Drive
    #    7z 분할 압축: TS.zip이 마지막 파트 (헤더 포함)
    #    단일 파일만 있을 때는 부분 추출 (이미지만)
    extract_dir = f'{DOWNLOAD_TMP}/extracted'
    os.makedirs(extract_dir, exist_ok=True)

    print(f'이미지 추출 중: {local_path}')
    ret = subprocess.run(
        ['7z', 'e', local_path, '-o', extract_dir, '*.jpg', '*.jpeg', '*.png', '-y'],
        capture_output=True, text=True
    )
    if ret.returncode != 0:
        print(f'⚠️  7z 추출 경고: {ret.stderr[:200]}')

    # 3. 추출된 이미지 → Drive 이동
    extracted_imgs = list(Path(extract_dir).rglob('*.jpg')) + \
                     list(Path(extract_dir).rglob('*.jpeg')) + \
                     list(Path(extract_dir).rglob('*.png'))
    print(f'추출된 이미지: {len(extracted_imgs):,}장 → Drive로 이동 중')

    import shutil
    for img_path in extracted_imgs:
        dst = Path(TRAIN_IMG_DIR) / img_path.name
        if not dst.exists():
            shutil.move(str(img_path), str(dst))

    # 4. 로컬 임시 파일 삭제 (디스크 확보)
    if os.path.exists(local_path):
        os.remove(local_path)
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)

    total = len(list(Path(TRAIN_IMG_DIR).rglob('*.jpg')))
    print(f'✅ {file_name} 처리 완료 | Drive train_images 누적: {total:,}장')

    # 디스크 여유 확인
    result = subprocess.run(['df', '-BG', '/content'], capture_output=True, text=True)
    print(result.stdout)


print('process_one_part 함수 준비 완료')
print('아래 셀에서 파트 번호를 선택해 순서대로 실행하세요')

In [ ]:
# ─────────────────────────────────────────────────────
# 분할 파일 순서대로 처리 (한 번에 1개씩)
# 디스크 부족 시 이 셀을 여러 번 나눠 실행
# ─────────────────────────────────────────────────────

# 처리할 파트 선택 (인덱스 0~7)
PART_INDEX = 0   # ← 0부터 순서대로 바꿔가며 실행

file_name, file_key = TRAIN_PARTS[PART_INDEX]
print(f'처리 대상: {file_name} (key={file_key}) [{PART_INDEX+1}/{len(TRAIN_PARTS)}]')
process_one_part(file_name, file_key)

In [ ]:
# ─────────────────────────────────────────────────────
# [선택] VS.zip — 검증 이미지 (88.80 GB)
# Drive 여유 공간 충분할 때만 실행
# ─────────────────────────────────────────────────────

# Drive 여유 공간 확인
result = subprocess.run(['df', '-BG', '/content/drive/MyDrive'], capture_output=True, text=True)
print('Drive 공간:')
print(result.stdout)

# 90GB 이상 여유 있을 때만 실행
DOWNLOAD_VS = False   # ← True로 바꾸면 다운로드 시작

if DOWNLOAD_VS:
    vs_zip_path = download_aihub_file(
        file_name='VS.zip',
        file_key=FILE_KEYS['VS.zip'],
        save_dir=DOWNLOAD_TMP,
        api_key=AIHUB_API_KEY,
    )
    vs_extract_dir = f'{DATASET_DIR}/val_images'
    os.makedirs(vs_extract_dir, exist_ok=True)
    print('VS.zip 압축 해제 중...')
    subprocess.run(['7z', 'e', vs_zip_path, '-o', vs_extract_dir, '*.jpg', '*.jpeg', '*.png', '-y'])
    img_count = len(list(Path(vs_extract_dir).rglob('*.jpg')))
    print(f'✅ 검증 이미지 {img_count:,}장 추출')
else:
    print('VS.zip 다운로드 스킵 — 검증은 학습 데이터 80:20 분리로 대체')

In [ ]:
# ─────────────────────────────────────────────────────
# 다운로드 현황 요약
# ─────────────────────────────────────────────────────
train_imgs = len(list(Path(TRAIN_IMG_DIR).rglob('*.jpg'))) if os.path.exists(TRAIN_IMG_DIR) else 0
val_jsons  = len(list(Path(f'{DATASET_DIR}/val_labels').rglob('*.json'))) if os.path.exists(f'{DATASET_DIR}/val_labels') else 0

print('=== 데이터 준비 현황 ===')
print(f'학습 이미지: {train_imgs:,}장  ({TRAIN_IMG_DIR})')
print(f'검증 라벨:   {val_jsons:,}개   ({DATASET_DIR}/val_labels)')
print()
if train_imgs == 0:
    print('⚠️  학습 이미지 없음 — PART_INDEX 0부터 순서대로 process_one_part 실행 필요')
elif train_imgs < 50000:
    print(f'⚠️  학습 이미지 {train_imgs:,}장 — 더 많은 분할 파일 처리 권장 (목표: 232,087장)')
else:
    print(f'✅ 학습 이미지 {train_imgs:,}장 확보 — Step 1 진행 가능')

# Step 1에서 사용할 경로 통합 설정
# 학습 이미지가 없으면 VL.zip(검증 라벨)만으로 클래스 수집
LABEL_DIR = f'{DATASET_DIR}/val_labels' if train_imgs == 0 else DATASET_DIR
print(f'\nStep 1 스캔 대상: {LABEL_DIR}')

## Step 1 — AI Hub 전체 클래스 수집

> AI Hub JSON에서 음식명(`food_type.fc`)을 전부 읽어 **클래스 목록을 자동으로 구성**한다.  
> 800종 전체를 학습 대상으로 사용.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

def collect_all_classes(dataset_dir: str) -> list:
    """
    AI Hub JSON 전체를 스캔하여 존재하는 음식명 목록을 수집.
    → 800종 전체를 클래스로 등록
    """
    json_files = list(Path(dataset_dir).rglob('*.json'))
    print(f'JSON 파일 총계: {len(json_files):,}개')

    class_count = defaultdict(int)
    error_count = 0

    for jf in json_files:
        try:
            with open(jf, encoding='utf-8') as f:
                data = json.load(f)
            fc = data.get('data', {}).get('food_type', {}).get('fc', '').strip()
            if fc:
                class_count[fc] += 1
        except Exception:
            error_count += 1

    # 이미지 수 기준 내림차순 정렬 → class_id 부여
    sorted_classes = sorted(class_count.items(), key=lambda x: -x[1])
    class_names = [cls for cls, _ in sorted_classes]

    print(f'\n발견된 클래스 수: {len(class_names)}종')
    print(f'파싱 오류: {error_count}개')
    print(f'\n=== 상위 20개 클래스 (이미지 수 기준) ===')
    for cls, cnt in sorted_classes[:20]:
        print(f'  {cls}: {cnt}장')

    return class_names, dict(class_count)


ALL_CLASSES, CLASS_COUNT = collect_all_classes(LABEL_DIR)
CLASS_TO_ID = {cls: idx for idx, cls in enumerate(ALL_CLASSES)}
print(f'\n총 클래스: {len(ALL_CLASSES)}종 → data.yaml nc={len(ALL_CLASSES)}')

## Step 2 — AI Hub JSON → YOLO 포맷 변환 (전체 변환)

> **스킵 없음** — 모든 이미지와 라벨을 YOLO 포맷으로 변환한다.  
> 변환 실패(bbox 오류, 이미지 없음)만 제외.

In [ ]:
import shutil
import random

# ─────────────────────────────────────────────────────
# AI Hub JSON 라벨 구조
# {
#   "data": {
#     "image_info":    { "file_name": "xxx.jpg", "width": 1280, "height": 960 },
#     "2d_annotation": { "x": 100, "y": 200, "width": 300, "height": 250 },
#     "food_type":     { "fc": "비빔밥" }
#   }
# }
#
# YOLO 포맷: class_id cx cy w h  (0~1 정규화, 중심 좌표)
# 변환: cx = (x + bw/2) / img_w   cy = (y + bh/2) / img_h
# ─────────────────────────────────────────────────────

def convert_one(json_path: str) -> tuple:
    """
    AI Hub JSON 1개 → (class_id, yolo_label, img_path)
    변환 불가 시 None 반환
    """
    try:
        with open(json_path, encoding='utf-8') as f:
            raw = json.load(f)
        data = raw.get('data', {})

        fc = data.get('food_type', {}).get('fc', '').strip()
        if not fc or fc not in CLASS_TO_ID:
            return None

        img_info = data.get('image_info', {})
        img_w    = float(img_info.get('width', 0))
        img_h    = float(img_info.get('height', 0))
        fname    = img_info.get('file_name', '')
        if img_w <= 0 or img_h <= 0 or not fname:
            return None

        ann = data.get('2d_annotation', {})
        x   = float(ann.get('x', 0))
        y   = float(ann.get('y', 0))
        bw  = float(ann.get('width', 0))
        bh  = float(ann.get('height', 0))
        if bw <= 0 or bh <= 0:
            return None

        cx = (x + bw / 2) / img_w
        cy = (y + bh / 2) / img_h
        nw = bw / img_w
        nh = bh / img_h

        if not all(0.0 < v <= 1.0 for v in [cx, cy, nw, nh]):
            return None

        class_id   = CLASS_TO_ID[fc]
        yolo_label = f'{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}'

        # 이미지 경로: JSON과 같은 폴더 → TRAIN_IMG_DIR 순으로 탐색
        img_path = Path(json_path).parent / fname
        if not img_path.exists():
            img_path = Path(TRAIN_IMG_DIR) / fname
        if not img_path.exists():
            candidates = list(Path(DATASET_DIR).rglob(fname))
            if not candidates:
                return None
            img_path = candidates[0]

        return class_id, yolo_label, str(img_path), fc

    except Exception:
        return None


def build_full_dataset(
    dataset_dir: str,
    output_dir: str,
    val_ratio: float = 0.2,
):
    """
    AI Hub 전체 데이터 → YOLO 데이터셋 구성 (클래스 필터링 없음)
    train:val = 8:2 분리
    """
    json_files = list(Path(dataset_dir).rglob('*.json'))
    print(f'변환 대상 JSON: {len(json_files):,}개')

    class_records = defaultdict(list)
    skip_count = 0

    for i, jf in enumerate(json_files):
        if i % 10000 == 0:
            print(f'  진행: {i:,} / {len(json_files):,}', end='\r')
        result = convert_one(str(jf))
        if result is None:
            skip_count += 1
            continue
        class_id, yolo_label, img_path, class_name = result
        class_records[class_name].append((yolo_label, img_path))

    print(f'\n\n변환 성공: {sum(len(v) for v in class_records.values()):,}장')
    print(f'변환 실패(bbox 오류/이미지 없음): {skip_count:,}개')

    print(f'\n=== 클래스별 이미지 수 (하위 10개) ===')
    sorted_records = sorted(class_records.items(), key=lambda x: len(x[1]))
    for cls, recs in sorted_records[:10]:
        status = '⚠️ ' if len(recs) < 50 else '  '
        print(f'{status} {cls}: {len(recs)}장')

    train_count = val_count = 0
    for class_name, records in class_records.items():
        random.shuffle(records)
        split     = max(1, int(len(records) * (1 - val_ratio)))
        train_set = records[:split]
        val_set   = records[split:]

        for split_name, split_recs in [('train', train_set), ('val', val_set)]:
            for yolo_label, img_src in split_recs:
                img_src  = Path(img_src)
                dst_img  = Path(output_dir) / split_name / 'images' / img_src.name
                dst_lbl  = Path(output_dir) / split_name / 'labels' / (img_src.stem + '.txt')
                shutil.copy2(str(img_src), str(dst_img))
                dst_lbl.write_text(yolo_label + '\n', encoding='utf-8')
                if split_name == 'train':
                    train_count += 1
                else:
                    val_count += 1

    print(f'\n=== 데이터셋 구성 완료 ===')
    print(f'train: {train_count:,}장')
    print(f'val:   {val_count:,}장')
    return train_count, val_count


train_count, val_count = build_full_dataset(
    dataset_dir=DATASET_DIR,
    output_dir=FINETUNE_DIR,
    val_ratio=0.2,
)

## Step 3 — data.yaml 생성

In [ ]:
import yaml

data_yaml_content = {
    'train': f'{FINETUNE_DIR}/train/images',
    'val':   f'{FINETUNE_DIR}/val/images',
    'nc':    len(ALL_CLASSES),
    'names': ALL_CLASSES,
}

yaml_path = f'{FINETUNE_DIR}/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml_content, f, allow_unicode=True, default_flow_style=False)

print(f'data.yaml 생성 완료')
print(f'  nc:    {len(ALL_CLASSES)}  (전체 클래스 수)')
print(f'  train: {FINETUNE_DIR}/train/images')
print(f'  val:   {FINETUNE_DIR}/val/images')
print(f'\n상위 10개 클래스:')
for i, cls in enumerate(ALL_CLASSES[:10]):
    print(f'  {i:3d}: {cls}')

## Step 4 — 파인튜닝 학습

> base 모델: `yolo11s.pt` (현재 프로젝트 `detector.py` 기본값과 동일)  
> 학습 파라미터: `detection_config.yaml` conf/iou 설정값 동일 적용

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

print('=== 학습 시작 ===')
print(f'base 모델:  yolo11s.pt')
print(f'클래스 수:  {len(ALL_CLASSES)}종')
print(f'data.yaml:  {FINETUNE_DIR}/data.yaml')
print(f'결과 저장:  {RUNS_DIR}/korean_food_v1/')
print()

results = model.train(
    data=f'{FINETUNE_DIR}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=20,
    project=RUNS_DIR,
    name='korean_food_v1',
    exist_ok=True,
    save=True,
    save_period=10,
    conf=0.001,
    iou=0.45,
    max_det=100,
    verbose=True,
)

print(f'\n=== 학습 완료 ===')
print(f'best.pt: {RUNS_DIR}/korean_food_v1/weights/best.pt')

## Step 5 — 검증 및 성능 확인

In [ ]:
from ultralytics import YOLO
import pandas as pd

best_pt = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'
model_ft = YOLO(best_pt)

metrics = model_ft.val(
    data=f'{FINETUNE_DIR}/data.yaml',
    device=0,
    conf=0.5,
    iou=0.45,
)

print('\n=== 전체 검증 결과 ===')
print(f'mAP@0.5:      {metrics.box.map50:.4f}  (목표: ≥ 0.60)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')

KEY_CLASSES = [
    '굴비', '조기', '대게', '전복', '장어', '한우', '흑돼지',
    '비빔밥', '김치찌개', '된장찌개', '삼겹살', '불고기',
    '사과', '딸기', '수박', '감귤',
]

print('\n=== 주요 클래스 AP@0.5 ===')
if hasattr(metrics.box, 'ap_class_index'):
    rows = []
    for i, class_idx in enumerate(metrics.box.ap_class_index):
        cls_name = ALL_CLASSES[int(class_idx)] if int(class_idx) < len(ALL_CLASSES) else str(class_idx)
        if cls_name not in KEY_CLASSES:
            continue
        ap50 = float(metrics.box.ap50[i]) if hasattr(metrics.box, 'ap50') else 0.0
        rows.append({'클래스': cls_name, 'AP@0.5': round(ap50, 4)})
    if rows:
        df = pd.DataFrame(rows).sort_values('AP@0.5', ascending=False)
        print(df.to_string(index=False))

if metrics.box.map50 >= 0.60:
    print(f'\n✅ mAP@0.5 = {metrics.box.map50:.4f} → 합격')
else:
    print(f'\n⚠️  mAP@0.5 = {metrics.box.map50:.4f} → 미달 — 데이터 추가 또는 epochs 증가 권장')

## Step 6 — A/B 비교 (기존 yolo11s.pt vs 파인튜닝 best.pt)

In [ ]:
from ultralytics import YOLO
import glob
from collections import Counter

val_images = glob.glob(f'{FINETUNE_DIR}/val/images/*.jpg')[:20]

if not val_images:
    print('val 이미지 없음 — Step 2 완료 후 실행')
else:
    model_base = YOLO('yolo11s.pt')
    model_ft   = YOLO(f'{RUNS_DIR}/korean_food_v1/weights/best.pt')

    base_labels = Counter()
    ft_labels   = Counter()

    for img_path in val_images:
        for pred in model_base(img_path, conf=0.5, verbose=False):
            for box in pred.boxes:
                base_labels[pred.names[int(box.cls)]] += 1
        for pred in model_ft(img_path, conf=0.5, verbose=False):
            for box in pred.boxes:
                ft_labels[pred.names[int(box.cls)]] += 1

    print(f'=== A/B 비교 (val 샘플 {len(val_images)}장) ===')
    print('\n기존 yolo11s.pt 탐지 상위 10개:')
    for label, cnt in base_labels.most_common(10):
        print(f'  {label}: {cnt}건')
    print('\n파인튜닝 best.pt 탐지 상위 10개:')
    for label, cnt in ft_labels.most_common(10):
        print(f'  {label}: {cnt}건')

    KEY_SET = set(KEY_CLASSES)
    base_key = sum(cnt for lbl, cnt in base_labels.items() if lbl in KEY_SET)
    ft_key   = sum(cnt for lbl, cnt in ft_labels.items()   if lbl in KEY_SET)
    print(f'\n=== 주요 클래스 탐지 건수 ===')
    print(f'기존:    {base_key}건')
    print(f'파인튜닝: {ft_key}건  (+{ft_key - base_key}건)')
    print('\n✅ 파인튜닝 효과 확인' if ft_key > base_key else '\n⚠️  효과 미미 — 데이터 품질 확인 필요')

## Step 7 — best.pt 다운로드 및 로컬 적용 안내

In [ ]:
import os

best_pt_path = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'

if os.path.exists(best_pt_path):
    size_mb = os.path.getsize(best_pt_path) / 1024 / 1024
    print(f'✅ best.pt 확인')
    print(f'   경로: {best_pt_path}')
    print(f'   크기: {size_mb:.1f} MB')
    print()
    print('=' * 50)
    print('로컬 적용 방법')
    print('=' * 50)
    print()
    print('1. Drive에서 best.pt 다운로드')
    print('   내 드라이브/runs/korean_food_v1/weights/best.pt')
    print()
    print('2. 로컬 프로젝트에 저장')
    print('   Object_Detection/models/korean_food_v1_best.pt')
    print()
    print('3. config/detection_config.yaml 수정 (한 줄만)')
    print('   변경 전: name: yolo11s.pt')
    print('   변경 후: name: models/korean_food_v1_best.pt')
    print()
    print('4. 기존 테스트 확인')
    print('   python -m pytest tests/test_phase1_setup.py -v')
    print('   → 13/13 PASS 필수')
    print()
    print('⚠️  models/*.pt 는 .gitignore 대상 — 커밋 금지')
else:
    print(f'⚠️  {best_pt_path} 없음 — Step 4 완료 후 재실행')

---
## 현재 파이프라인 연동 포인트

```
Object_Detection/
├── src/detector.py              ← Detector(model_name=...) 모델 경로만 변경
├── config/detection_config.yaml ← name: models/korean_food_v1_best.pt
├── models/
│   └── korean_food_v1_best.pt   ← .gitignore 대상
└── data/
    └── vod_detected_object.parquet  ← 800종 한식 라벨 포함 재생성
```

### parquet 스키마 변경 없음

| 컬럼 | 타입 | 변화 |
|------|------|------|
| `vod_id` | str | 동일 |
| `frame_ts` | float | 동일 |
| `label` | str | **800종 한식 라벨 추가** (비빔밥, 굴비, 대게 등) |
| `confidence` | float | 동일 |
| `bbox` | list[float] | 동일 |

Shopping_Ad 연동 코드 변경 없이 `label` 다양성만 증가.